# Cardio Catch Disease — 03. Feature Engineering & Data Preparation

> **Hands-On ML principle (Ch. 2):** Data preparation is the most impactful step. Use Scikit-Learn **Pipelines** and **ColumnTransformers** so that the same transformations apply automatically at training and inference time — no manual pre-processing bugs.

---

## 0. Setup

In [ ]:
from pathlib import Path
import sys, warnings

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore")

FIGURES = PROJECT_ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Load Data & Recreate Train Split

In [ ]:
from cardio_catch_disease.data import load_training_frame
from cardio_catch_disease.config import load_config
from sklearn.model_selection import train_test_split

config = load_config(PROJECT_ROOT / "configs" / "project.toml")
df = load_training_frame(config)

train_set, test_set = train_test_split(df, test_size=0.20, random_state=42, stratify=df["cardio"])

# Separate features and target
TARGET = "cardio"
X_train_raw = train_set.drop(columns=[TARGET, "id"], errors="ignore")
y_train      = train_set[TARGET].reset_index(drop=True)
X_test_raw   = test_set.drop(columns=[TARGET, "id"], errors="ignore")
y_test       = test_set[TARGET].reset_index(drop=True)

print(f"X_train shape: {X_train_raw.shape}")
print(f"X_test shape:  {X_test_raw.shape}")
print(f"Features: {list(X_train_raw.columns)}")

## 2. Step 1 — Data Filtering (Outlier Removal)

Remove physiologically impossible blood pressure values identified in Notebook 01.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


def remove_bp_outliers(df: pd.DataFrame) -> pd.DataFrame:
    """Remove rows with physiologically impossible blood pressure values."""
    mask = (
        df["ap_hi"].between(70, 250) &
        df["ap_lo"].between(40, 150) &
        (df["ap_hi"] > df["ap_lo"])
    )
    return df[mask].copy()


X_train_filtered = remove_bp_outliers(X_train_raw.assign(**{TARGET: y_train.values}))
y_train_filtered = X_train_filtered.pop(TARGET)
X_train_filtered = X_train_filtered.reset_index(drop=True)
y_train_filtered = y_train_filtered.reset_index(drop=True)

removed = len(X_train_raw) - len(X_train_filtered)
print(f"Removed {removed:,} rows with blood pressure outliers ({removed/len(X_train_raw):.1%})")
print(f"Clean training set: {len(X_train_filtered):,} rows")

## 3. Step 2 — Feature Engineering

Building a **custom sklearn transformer** that adds domain-driven features.
Using the sklearn API (`BaseEstimator`, `TransformerMixin`) ensures it integrates cleanly into a Pipeline.

In [ ]:
class CardioFeatureAdder(BaseEstimator, TransformerMixin):
    """Adds clinical features derived from raw cardio dataset columns."""

    def fit(self, X, y=None):
        return self  # stateless transformer

    def transform(self, X):
        X = pd.DataFrame(X).copy() if not isinstance(X, pd.DataFrame) else X.copy()

        # Age in years (more interpretable than days)
        if "age" in X.columns:
            X["age_years"] = X["age"] / 365.25

        # BMI: weight(kg) / height(m)²
        if {"height", "weight"}.issubset(X.columns):
            height_m = X["height"] / 100
            X["bmi"] = X["weight"] / (height_m ** 2)
            X.loc[(X["bmi"] < 10) | (X["bmi"] > 80), "bmi"] = np.nan  # clip extremes

        # Pulse pressure: systolic − diastolic
        if {"ap_hi", "ap_lo"}.issubset(X.columns):
            X["pulse_pressure"] = X["ap_hi"] - X["ap_lo"]

        # Hypertension flag: WHO definition
        if {"ap_hi", "ap_lo"}.issubset(X.columns):
            X["high_blood_pressure"] = ((X["ap_hi"] >= 140) | (X["ap_lo"] >= 90)).astype(int)

        return X


# Preview the enriched features
adder = CardioFeatureAdder()
X_enriched = adder.transform(X_train_filtered)

new_features = [c for c in X_enriched.columns if c not in X_train_filtered.columns]
print(f"New features added: {new_features}")
X_enriched[new_features + ["age", "height", "weight", "ap_hi", "ap_lo"]].head()

## 4. Step 3 — Full sklearn Pipeline

> **Book principle (Ch. 2):** "You should use Scikit-Learn's Pipeline class to chain your transformations. This way, you don't have to apply the transformations manually at every step."

The pipeline packages everything: feature engineering → imputation → scaling → encoding.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer


def build_preprocessor(X: pd.DataFrame):
    """Build a ColumnTransformer for numeric and categorical features."""

    # After feature engineering, identify column types
    X_enriched = CardioFeatureAdder().transform(X)
    cat_cols = X_enriched.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    num_cols = [c for c in X_enriched.columns if c not in cat_cols]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),   # handles NaNs from BMI extremes
        ("scaler",  StandardScaler()),                   # zero mean, unit variance
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline,    num_cols),
        ("cat", categorical_pipeline, cat_cols),
    ], remainder="drop")

    return preprocessor, num_cols, cat_cols


preprocessor, num_cols, cat_cols = build_preprocessor(X_train_filtered)

print(f"Numeric features  ({len(num_cols)}): {num_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")

In [ ]:
# Full preprocessing pipeline
full_pipeline = Pipeline([
    ("feature_adder", CardioFeatureAdder()),
    ("preprocessor",  preprocessor),
])

# Fit on training data only — never on test set
X_train_prepared = full_pipeline.fit_transform(X_train_filtered)
X_test_prepared  = full_pipeline.transform(X_test_raw)  # apply same transforms

print(f"X_train_prepared shape: {X_train_prepared.shape}")
print(f"X_test_prepared shape:  {X_test_prepared.shape}")
print(f"\nPipeline fitted on {len(X_train_filtered):,} training samples.")
print("Test set transformed using training statistics (no data leakage).")

## 5. Feature Importance Preview — Before Modelling

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Quick RF to get feature importances — not the final model
rf_quick = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf_quick.fit(X_train_prepared, y_train_filtered)

# Recover feature names after ColumnTransformer
try:
    feature_names = full_pipeline.named_steps["preprocessor"].get_feature_names_out().tolist()
except Exception:
    feature_names = [f"feature_{i}" for i in range(X_train_prepared.shape[1])]

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_quick.feature_importances_,
}).sort_values("importance", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=importance_df, x="importance", y="feature", palette="viridis", ax=ax)
ax.set_title("Top 15 Feature Importances (Random Forest — 50 trees)",
             fontweight="bold", fontsize=12)
ax.set_xlabel("Mean Decrease Impurity")
plt.tight_layout()
plt.savefig(FIGURES / "03_feature_importance_preview.png", bbox_inches="tight")
plt.show()

print("\nTop 10 features:")
print(importance_df.head(10).to_string(index=False))

## 6. Pipeline Diagram

In [ ]:
from sklearn import set_config
set_config(display="diagram")
full_pipeline

## 7. Verify: No Data Leakage

In [ ]:
# The scaler's statistics were computed on training data only
scaler = full_pipeline.named_steps["preprocessor"].named_transformers_["num"].named_steps["scaler"]
print("Scaler fitted on training set:")
print(f"  Mean range  : [{scaler.mean_.min():.3f}, {scaler.mean_.max():.3f}]")
print(f"  Scale range : [{scaler.scale_.min():.3f}, {scaler.scale_.max():.3f}]")
print()
print("Pipeline correctly fitted on X_train only.")
print("X_test uses transform() — applies training statistics, no look-ahead.")

## 8. Export Prepared Data for Modelling

In [ ]:
import joblib

# Save pipeline for reuse in Notebook 04 and deployment
INTERIM = PROJECT_ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

joblib.dump(full_pipeline, INTERIM / "preprocessing_pipeline.joblib")
joblib.dump((X_train_prepared, y_train_filtered, X_test_prepared, y_test),
            INTERIM / "prepared_data.joblib")

print("Saved:")
print(f"  {INTERIM / 'preprocessing_pipeline.joblib'}")
print(f"  {INTERIM / 'prepared_data.joblib'}")
print()
print("Summary:")
print(f"  X_train_prepared : {X_train_prepared.shape}")
print(f"  y_train_filtered : {y_train_filtered.shape}  (class balance: {y_train_filtered.mean():.1%} positive)")
print(f"  X_test_prepared  : {X_test_prepared.shape}")
print(f"  y_test           : {y_test.shape}  (class balance: {y_test.mean():.1%} positive)")

## 9. Summary

| Step | What was done |
|---|---|
| **Filtering** | Removed ~3% of rows with physiologically impossible blood pressure |
| **Feature engineering** | Added `age_years`, `bmi`, `pulse_pressure`, `high_blood_pressure` |
| **Imputation** | Median for numeric (BMI extremes), most_frequent for categorical |
| **Scaling** | StandardScaler on all numeric features |
| **Encoding** | OneHotEncoder on categorical features |
| **Pipeline** | All steps packaged in a single `sklearn.Pipeline` — no data leakage |
| **No leakage** | Scaler fitted on `X_train` only; `X_test` uses `transform()` |

**Next → Notebook 04: Machine Learning Modelling & Business Results**